# Road Safety - Data Cleaning and Clustering

Cleaning the WHO road safety file and running K-Means on it. Output goes to `data/processed/` so the rest of the project can use it.

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')

### Load the file

The Excel file has two header rows. The first is short codes (CP_0, CP_1...) and the second is the actual column names. So we tell pandas to use row 1 as the header.

In [ ]:
RAW_PATH = 'data/raw/gsrrs23-indicators-for-participating-countries-or-territories.xlsx'

df = pd.read_excel(RAW_PATH, sheet_name='Indicators', header=1)
df = df.iloc[1:].reset_index(drop=True)  # drop leftover code row

df.shape

In [ ]:
df.head(3)

### First look

In [ ]:
df.info()

In [ ]:
df[['ISO_3 country name', 'Country name', 'Population', 'Income group', 'WHO Region']].head()

### Check for duplicates

In [ ]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate ISO codes:', df['ISO_3 country name'].duplicated().sum())
print('duplicate country names:', df['Country name'].duplicated().sum())

### Missing values

A lot of WHO indicators are missing for a lot of countries. Let's see how bad it is.

In [ ]:
miss_pct = df.isna().mean().sort_values(ascending=False) * 100

print('columns >50% missing:', (miss_pct > 50).sum())
print('columns >70% missing:', (miss_pct > 70).sum())
print('columns with 0 missing:', (miss_pct == 0).sum())
print('total columns:', len(df.columns))

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(miss_pct, bins=30, edgecolor='black')
plt.axvline(50, color='red', linestyle='--', label='50% cutoff')
plt.xlabel('% missing')
plt.ylabel('number of columns')
plt.title('How empty each column is')
plt.legend()
plt.show()

### Fix Palestine's name

The WHO file calls it "occupied Palestinian territory, including east Jerusalem". The other dataset (output.csv) just says "Palestine". Map them to the same thing.

In [ ]:
COUNTRY_NAME_MAP = {
    'occupied Palestinian territory, including east Jerusalem': 'Palestine',
    'occupied Palestinian territory': 'Palestine',
    'State of Palestine': 'Palestine',
    'Palestinian Territory': 'Palestine',
    'West Bank and Gaza Strip': 'Palestine',
}

df['Country name'] = df['Country name'].replace(COUNTRY_NAME_MAP)
df[df['ISO_3 country name'] == 'PSE'][['ISO_3 country name', 'Country name']]

### Drop columns that are mostly empty

Anything more than half empty gets dropped.

In [ ]:
cols_to_drop = miss_pct[miss_pct > 50].index.tolist()
df_clean = df.drop(columns=cols_to_drop)

print('before:', df.shape[1])
print('after: ', df_clean.shape[1])
print('dropped:', len(cols_to_drop))

### Pick the numeric columns

K-Means needs numbers, so split off the label columns (country, region, income group) from the numeric features.

In [ ]:
LABEL_COLS = ['ISO_3 country name', 'Country name', 'Income group', 'WHO Region']

numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
print('numeric features:', len(numeric_cols))

### Fill the remaining missing values

Using median because it's not pulled around by extreme values.

In [ ]:
features = df_clean[numeric_cols].copy()

print('missing before:', features.isna().sum().sum())
features = features.fillna(features.median(numeric_only=True))
print('missing after: ', features.isna().sum().sum())

### Scale the features

K-Means uses distance, so if Population is in the millions and seatbelt % is 0-100, population dominates everything. StandardScaler puts every column on mean 0, std 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print(X_scaled.shape)
print('first column mean:', round(X_scaled[:, 0].mean(), 4))
print('first column std: ', round(X_scaled[:, 0].std(), 4))

### Find a good k

Try k = 2 through 8 and look at inertia (elbow) and silhouette score.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

pd.DataFrame({'k': list(k_values), 'inertia': inertias, 'silhouette': silhouettes}).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(k_values), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k')
axes[0].set_ylabel('inertia')
axes[0].set_title('Elbow')

axes[1].plot(list(k_values), silhouettes, 'o-', color='darkorange')
axes[1].set_xlabel('k')
axes[1].set_ylabel('silhouette')
axes[1].set_title('Silhouette')

plt.tight_layout()
plt.show()

### Fit K-Means with the chosen k

Picking k = 4 based on the plots above. Change this if your silhouette peaks somewhere else.

In [ ]:
K_FINAL = 4

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

pd.Series(cluster_labels).value_counts().sort_index()

### Cluster scatter in 2D

Project the features down to 2 dimensions with PCA just so we can plot them. Each dot is a country, color is its cluster, Palestine is starred.

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
coords = pca_2d.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': cluster_labels,
    'Country': df_clean['Country name'].values,
    'ISO': df_clean['ISO_3 country name'].values,
})
plot_df.head()

In [ ]:
plt.figure(figsize=(12, 8))

palette = sns.color_palette('Set2', K_FINAL)
for c in range(K_FINAL):
    sub = plot_df[plot_df['Cluster'] == c]
    plt.scatter(sub['PC1'], sub['PC2'],
                color=palette[c], label=f'Cluster {c}',
                s=70, alpha=0.7, edgecolor='black', linewidth=0.5)

pal = plot_df[plot_df['ISO'] == 'PSE']
plt.scatter(pal['PC1'], pal['PC2'],
            marker='*', s=400, color='red',
            edgecolor='black', linewidth=1.5, label='Palestine', zorder=5)

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.title(f'Country clusters (k = {K_FINAL})')
plt.legend()
plt.tight_layout()
plt.show()

### What's in each cluster

In [ ]:
profile_df = df_clean[LABEL_COLS].copy()
profile_df['Cluster'] = cluster_labels

pd.crosstab(profile_df['Cluster'], profile_df['Income group'])

In [ ]:
pd.crosstab(profile_df['Cluster'], profile_df['WHO Region'])

### Top features separating the clusters

Which features differ the most across clusters - those are the ones K-Means is using to split countries.

In [ ]:
cluster_means = features.copy()
cluster_means['Cluster'] = cluster_labels
group_means = cluster_means.groupby('Cluster').mean()

variation = group_means.std().sort_values(ascending=False)
top_distinguishing = variation.head(10).index

group_means[top_distinguishing].round(2)

### Where Palestine lands

In [ ]:
pal_cluster = profile_df.loc[profile_df['ISO_3 country name'] == 'PSE', 'Cluster'].values[0]
print('Palestine cluster:', pal_cluster)

neighbors = profile_df[profile_df['Cluster'] == pal_cluster]['Country name'].tolist()
print('\nSame cluster:')
for n in neighbors:
    print(' -', n)

### Save the outputs

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

features_out = df_clean[LABEL_COLS].copy()
features_out = pd.concat([features_out, features.reset_index(drop=True)], axis=1)
features_out.to_csv('data/processed/country_features.csv', index=False)

clusters_out = profile_df.copy()
clusters_out.to_csv('data/processed/country_clusters.csv', index=False)

print('saved country_features.csv', features_out.shape)
print('saved country_clusters.csv', clusters_out.shape)